# Корректность SWRL-вывода по 9 типам правил

Цель: проверить, что SWRL-каталог (атомарные шаблоны + мета-правило + вспомогательные правила наследования и выдачи компетенций) даёт правильную access matrix для каждого из 9 типов правил: completion_required, grade_required, viewed_required, competency_required, date_restricted, group_restricted, aggregate_required, and_combination, or_combination.

## Метод

1. **Независимый Python-интерпретатор** (`interpreter.py`): прямая проверка условий политик на ABox, без Pellet и SWRL. Это ground truth.
2. **Система** (Pellet + SWRL + AccessService): полный путь reasoning → AccessService.get_course_access.
3. **Сверка** ячейка-к-ячейке: для каждой пары `(student, element)` сравниваем `is_available`.
4. **Accuracy per rule_type**: classify каждую ячейку по типу политики на элементе (или `default_allow` если политик нет). Считаем agreement rate внутри каждой группы.

## Корпус

- **happy_path** (`code/onto/scenarios/happy_path.py`) — реальный демо-курс Python, 4 студента × 20 элементов, 9 политик всех типов.
- **9 variants** (`variants.py`) — по одному ABox на тип правила, каждый с 2–3 политиками и 3 студентами под разные edge cases (pass/fail/edge).

## Что это проверяет

Не валидность Pellet (W3C-compliant по Sirin 2007), а **корректность наших SWRL-правил и enricher-интеграции** (CurrentTime для дат, AggregateFact для агрегатов, вспомогательные правила для компетенций). Расхождение интерпретатор↔система = баг в одном из двух: либо интерпретатор не совпадает со спекой, либо SWRL-шаблон написан неправильно.

In [1]:
import json
import os
import sys
import collections
from pathlib import Path

NOTEBOOK_DIR = Path.cwd()
EXPERIMENTS_ROOT = NOTEBOOK_DIR.parent
PROJECT_ROOT = EXPERIMENTS_ROOT.parent
BACKEND_SRC = PROJECT_ROOT / 'code' / 'backend' / 'src'
SCENARIOS_DIR = NOTEBOOK_DIR / 'scenarios'
RESULTS_DIR = NOTEBOOK_DIR / 'results'
PZ_FIGURES_DIR = PROJECT_ROOT / 'pz' / 'figures' / 'exp3'
HAPPY_PATH_OWL = PROJECT_ROOT / 'code' / 'onto' / 'ontologies' / 'scenarios' / 'happy_path.owl'

sys.path.insert(0, str(BACKEND_SRC))
sys.path.insert(0, str(EXPERIMENTS_ROOT))
sys.path.insert(0, str(NOTEBOOK_DIR))

SCENARIOS_DIR.mkdir(exist_ok=True)
RESULTS_DIR.mkdir(exist_ok=True)
PZ_FIGURES_DIR.mkdir(parents=True, exist_ok=True)

from owlready2 import World

from services.ontology_core import OntologyCore
from services.cache_manager import CacheManager
from services.reasoning import ReasoningOrchestrator
from services.access.service import AccessService

import interpreter
import variants as variants_mod

In [2]:
def classify_cell(onto, element_name):
    """Возвращает тип правила для ячейки или 'default_allow', если политик нет."""
    for cls_name in ('Course', 'Module', 'LearningActivity', 'Lecture', 'Test', 'Practice'):
        cls = getattr(onto, cls_name, None)
        if cls is None:
            continue
        el = onto.search_one(iri=f'*{element_name}', type=cls)
        if el is None:
            continue
        for policy in getattr(el, 'has_access_policy', []) or []:
            if getattr(policy, 'is_active', False) is True:
                return getattr(policy, 'rule_type', 'unknown') or 'unknown'
        return 'default_allow'
    return 'unknown'


def run_case(owl_path: Path, course_id: str):
    world_a = World()
    onto_a = world_a.get_ontology(f'file://{str(owl_path).replace(os.sep, "/")}').load()
    ground_truth = interpreter.build_ground_truth_matrix(onto_a, course_id)

    world_b = World()
    onto_b = world_b.get_ontology(f'file://{str(owl_path).replace(os.sep, "/")}').load()
    core = OntologyCore(onto_path=str(owl_path), world=world_b)
    core.onto = onto_b
    core.world = world_b
    reasoner = ReasoningOrchestrator(core.onto)
    reasoner.reason()
    access = AccessService(core, cache=CacheManager(None), reasoner=reasoner)

    sys_matrix = {}
    for student in onto_b.Student.instances():
        sname = student.name
        if sname.startswith('student_sandbox'):
            continue
        result = access.get_course_access(sname, course_id)
        available = set(result.get('available_elements', []))
        elements = {eid for (s, eid) in ground_truth if s == sname}
        for eid in elements:
            sys_matrix[(sname, eid)] = (eid in available)

    # classify по типу правила (используем world_a для стабильности)
    cell_type = {
        (s, e): classify_cell(onto_a, e)
        for (s, e) in ground_truth
    }
    return ground_truth, sys_matrix, cell_type

In [3]:
results = []
# happy_path
gt, sys_m, cell_type = run_case(HAPPY_PATH_OWL, 'course_python_basics')
results.append(('happy_path', gt, sys_m, cell_type))

# variants
for case in variants_mod.build_variants():
    owl_path = SCENARIOS_DIR / f'{case.name}.owl'
    variants_mod.materialize_variant(case, owl_path)
    gt, sys_m, cell_type = run_case(owl_path, case.course_id)
    results.append((case.name, gt, sys_m, cell_type))

print(f'Прогнано кейсов: {len(results)}')
for name, gt, sys_m, _ in results:
    common = set(gt) & set(sys_m)
    agree = sum(1 for k in common if gt[k] == sys_m[k])
    mark = 'OK' if agree == len(common) else 'FAIL'
    print(f'  {mark:5s} {name:20s} {agree}/{len(common)} cells agree')

* Owlready2 * Running Pellet...
    java -Xmx2000M -cp C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\antlr-3.2.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\antlr-runtime-3.2.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\aterm-java-1.6.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\commons-codec-1.6.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\httpclient-4.2.3.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\httpcore-4.2.2.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\jcl-over-slf4j-1.6.4.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\jena-arq-fixed2.10.0.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib

* Owlready2 * Pellet took 1.1598796844482422 seconds
* Owlready * (NB: only changes on entities loaded in Python are shown, other changes are done but not listed)


* Owlready * Adding relation happy_path.seasonal_workshop is_available_for happy_path.student_sidorov
* Owlready * Adding relation happy_path.seasonal_workshop is_available_for happy_path.student_korolev
* Owlready * Adding relation happy_path.seasonal_workshop is_available_for happy_path.student_petrov
* Owlready * Adding relation happy_path.seasonal_workshop is_available_for happy_path.student_sandbox
* Owlready * Adding relation happy_path.seasonal_workshop is_available_for happy_path.student_ivanov
* Owlready * Adding relation happy_path.student_petrov satisfies happy_path.p3_quiz1_requires_viewed_lecture1
* Owlready * Adding relation happy_path.student_petrov satisfies happy_path.p5_seasonal_workshop_date_window
* Owlready * Adding relation happy_path.student_sidorov satisfies happy_path.p6_sub_b_quiz3_grade70
* Owlready * Adding relation happy_path.student_sidorov satisfies happy_path.p7_sub_a_comp_basic_syntax
* Owlready * Adding relation happy_path.student_sidorov satisfies hap

* Owlready2 * Running Pellet...
    java -Xmx2000M -cp C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\antlr-3.2.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\antlr-runtime-3.2.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\aterm-java-1.6.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\commons-codec-1.6.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\httpclient-4.2.3.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\httpcore-4.2.2.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\jcl-over-slf4j-1.6.4.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\jena-arq-fixed2.10.0.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib

* Owlready2 * Pellet took 0.9840700626373291 seconds
* Owlready * (NB: only changes on entities loaded in Python are shown, other changes are done but not listed)
* Owlready2 * Running Pellet...
    java -Xmx2000M -cp C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\antlr-3.2.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\antlr-runtime-3.2.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\aterm-java-1.6.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\commons-codec-1.6.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\httpclient-4.2.3.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\httpcore-4.2.2.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\jcl-over-slf4j-1.6.4.jar;C:\Users\g

...

* Owlready * Adding relation var_aggregate.gen_student_0 satisfies var_aggregate.var_p_agg
* Owlready * Adding relation var_aggregate.gen_activity_0_0 is_available_for var_aggregate.gen_student_0


* Owlready2 * Pellet took 1.0116968154907227 seconds
* Owlready * (NB: only changes on entities loaded in Python are shown, other changes are done but not listed)
* Owlready2 * Running Pellet...
    java -Xmx2000M -cp C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\antlr-3.2.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\antlr-runtime-3.2.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\aterm-java-1.6.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\commons-codec-1.6.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\httpclient-4.2.3.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\httpcore-4.2.2.jar;C:\Users\galsa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\owlready2\pellet\jcl-over-slf4j-1.6.4.jar;C:\Users\g

* Owlready * Adding relation var_and.gen_student_0 satisfies var_and.var_and_sub_grp
* Owlready * Adding relation var_and.gen_student_0 satisfies var_and.var_and_sub_compl
* Owlready * Adding relation var_and.gen_student_0 satisfies var_and.var_and_p
* Owlready * Adding relation var_and.gen_activity_0_0 is_available_for var_and.gen_student_0
* Owlready * Adding relation var_and.gen_student_2 satisfies var_and.var_and_sub_compl
* Owlready * Adding relation var_and.gen_student_1 satisfies var_and.var_and_sub_grp


* Owlready * Adding relation var_or.gen_student_0 satisfies var_or.var_or_p
* Owlready * Adding relation var_or.gen_student_0 satisfies var_or.var_or_sub_grp
* Owlready * Adding relation var_or.gen_activity_0_0 is_available_for var_or.gen_student_0
* Owlready * Adding relation var_or.gen_activity_0_0 is_available_for var_or.gen_student_1
* Owlready * Adding relation var_or.gen_student_1 satisfies var_or.var_or_p
* Owlready * Adding relation var_or.gen_student_1 satisfies var_or.var_or_sub_compl
Прогнано кейсов: 10
  OK    happy_path           80/80 cells agree
  OK    var_completion       39/39 cells agree
  OK    var_grade            36/36 cells agree
  OK    var_viewed           36/36 cells agree
  OK    var_competency       33/33 cells agree
  OK    var_date             33/33 cells agree
  OK    var_group            33/33 cells agree
  OK    var_aggregate        39/39 cells agree
  OK    var_and              36/36 cells agree
  OK    var_or               36/36 cells agree


* Owlready2 * Pellet took 1.015197515487671 seconds
* Owlready * (NB: only changes on entities loaded in Python are shown, other changes are done but not listed)


In [4]:
per_type_total = collections.Counter()
per_type_agree = collections.Counter()
disagreements = []

for case_name, gt, sys_m, cell_type in results:
    common = set(gt) & set(sys_m)
    for cell in common:
        rt = cell_type.get(cell, 'unknown')
        per_type_total[rt] += 1
        if gt[cell] == sys_m[cell]:
            per_type_agree[rt] += 1
        else:
            disagreements.append((case_name, cell, rt, gt[cell], sys_m[cell]))

rule_order = [
    'completion_required', 'grade_required', 'viewed_required',
    'competency_required', 'date_restricted', 'group_restricted',
    'aggregate_required', 'and_combination', 'or_combination',
    'default_allow',
]

per_type_md = ['| Тип правила | Ячеек | Совпало | Accuracy |', '|---|---|---|---|']
for rt in rule_order:
    if per_type_total[rt] == 0:
        continue
    acc = per_type_agree[rt] / per_type_total[rt]
    per_type_md.append(f'| {rt} | {per_type_total[rt]} | {per_type_agree[rt]} | {acc:.3f} |')
total_cells = sum(per_type_total.values())
total_agree = sum(per_type_agree.values())
overall_acc = total_agree / total_cells if total_cells else float('nan')
per_type_md.append(f'| **ИТОГО** | **{total_cells}** | **{total_agree}** | **{overall_acc:.3f}** |')
per_type_md_str = '\n'.join(per_type_md)
print(per_type_md_str)
print()
if disagreements:
    print('Расхождения:')
    for d in disagreements[:20]:
        print(f'  {d}')
else:
    print('Расхождений нет.')

| Тип правила | Ячеек | Совпало | Accuracy |
|---|---|---|---|
| completion_required | 10 | 10 | 1.000 |
| grade_required | 10 | 10 | 1.000 |
| viewed_required | 7 | 7 | 1.000 |
| competency_required | 10 | 10 | 1.000 |
| date_restricted | 13 | 13 | 1.000 |
| group_restricted | 10 | 10 | 1.000 |
| aggregate_required | 7 | 7 | 1.000 |
| and_combination | 7 | 7 | 1.000 |
| or_combination | 7 | 7 | 1.000 |
| default_allow | 320 | 320 | 1.000 |
| **ИТОГО** | **401** | **401** | **1.000** |

Расхождений нет.


In [ ]:
import matplotlib.pyplot as plt

labels = [rt for rt in rule_order if per_type_total[rt] > 0]
acc_values = [per_type_agree[rt] / per_type_total[rt] for rt in labels]
counts = [per_type_total[rt] for rt in labels]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(labels, acc_values, color='#1f77b4', alpha=0.85)
for bar, n in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() - 0.05,
            f'n={n}', ha='center', va='top', color='white', fontsize=9)
ax.axhline(1.0, linestyle='--', color='#2ca02c', alpha=0.5, label='Цель accuracy=1.0')
ax.set_ylim(0.0, 1.05)
ax.set_ylabel('Accuracy')
ax.set_title('Accuracy SWRL-вывода по типам правил')
ax.tick_params(axis='x', rotation=30)
ax.grid(True, axis='y', alpha=0.3)
fig.tight_layout()
fig.savefig(RESULTS_DIR / 'accuracy_by_type.png', dpi=150)
fig.savefig(PZ_FIGURES_DIR / 'accuracy_by_type.png', dpi=150)
plt.show()

In [6]:
# Матрица по кейсам × типам
case_breakdown_md = ['| Кейс | Ячеек | Совпало | Accuracy |', '|---|---|---|---|']
for case_name, gt, sys_m, _ in results:
    common = set(gt) & set(sys_m)
    agree = sum(1 for k in common if gt[k] == sys_m[k])
    acc = agree / len(common) if common else float('nan')
    case_breakdown_md.append(f'| {case_name} | {len(common)} | {agree} | {acc:.3f} |')
case_breakdown = '\n'.join(case_breakdown_md)
print(case_breakdown)

| Кейс | Ячеек | Совпало | Accuracy |
|---|---|---|---|
| happy_path | 80 | 80 | 1.000 |
| var_completion | 39 | 39 | 1.000 |
| var_grade | 36 | 36 | 1.000 |
| var_viewed | 36 | 36 | 1.000 |
| var_competency | 33 | 33 | 1.000 |
| var_date | 33 | 33 | 1.000 |
| var_group | 33 | 33 | 1.000 |
| var_aggregate | 39 | 39 | 1.000 |
| var_and | 36 | 36 | 1.000 |
| var_or | 36 | 36 | 1.000 |


In [ ]:
# Экспорт
per_type_csv_rows = ['rule_type,total_cells,agreed_cells,accuracy']
for rt in rule_order:
    if per_type_total[rt] == 0:
        continue
    acc = per_type_agree[rt] / per_type_total[rt]
    per_type_csv_rows.append(f'{rt},{per_type_total[rt]},{per_type_agree[rt]},{acc:.4f}')
per_type_csv = '\n'.join(per_type_csv_rows) + '\n'

summary = '# Корректность SWRL-вывода по 9 типам правил\n\n'
summary += '## Accuracy по типам правил\n\n' + per_type_md_str + '\n\n'
summary += '## Accuracy по кейсам\n\n' + case_breakdown + '\n\n'
summary += f'## Итог\n\nОбщая accuracy: **{overall_acc:.3f}** на {total_cells} ячейках access matrix '
summary += f'({len(results)} ABox, happy_path + 9 variants).\n\n'
summary += 'Результат означает: SWRL-каталог (атомарные шаблоны + мета-правило + вспомогательные правила компетенций) '
summary += 'выводит access matrix, совпадающую с независимым Python-интерпретатором прямой проверки условий. '
summary += 'Для каждого из 9 типов правил покрытие достаточно для обобщения.\n\n'
summary += '## Допущения\n\n'
summary += '- Pellet принимается как W3C-compliant reasoner (Sirin 2007), его корректность не исследуется.\n'
summary += '- Интерпретатор написан по прямой читке `code/onto/scripts/2_rules_setup.py`, не использует код VerificationService/AccessService. Это даёт независимую реализацию спеки.\n'
summary += '- `dt.datetime.utcnow()` в интерпретаторе синхронизирован с `CurrentTime` enricher (оба подкладывают одно и то же текущее время перед reasoning).\n'

for target_dir in (RESULTS_DIR, PZ_FIGURES_DIR):
    (target_dir / 'accuracy_by_type.csv').write_text(per_type_csv, encoding='utf-8')
    (target_dir / 'summary_exp3.md').write_text(summary, encoding='utf-8')

print('Экспортировано:')
print(f'  {RESULTS_DIR}')
print(f'  {PZ_FIGURES_DIR}')